> **Snap Market model series.** We start from Carl's baseline momentum model and iterate to test it and improve it.
>
> **Notebook 10 — the business view.** With elastic demand (better odds draw more volume), compare which model makes the most net house PnL, and show how the answer depends on demand elasticity.

# 10 — Which model makes the most money, and how it depends on demand elasticity

Profit = house edge times volume, and volume depends on how competitive the odds are. We compare
**every model** under **elastic demand** (`demand_responsive_pool`: tighter odds draw more
volume) with the two expected-value-gated arbitrageurs (predictive, regime-aware) betting inside
the pool, and read off the **net house PnL**.

The one free assumption is the **demand elasticity** — how strongly volume reacts to the offered
margin. We do not know it, so we sweep it from 2 to 32 and show how the ranking moves. Use the
slider below.

Why the vig-only models (`momentum_lookup`, `momentum_lookup_rolling`, `volatility_regime_momentum`,
`momentum_logistic_rolling`) are roughly flat in elasticity: they all quote the same margin, so the
elastic pool reacts the same way for all of them. Only the models that charge an extra margin
(`hidden_symmetric_margin` / model 3, and `guarded_volatility_regime_momentum`) lose volume as
elasticity rises.

In [ ]:
import os
import sys

# Move up one level to the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
# Add the new current directory to the Python path
sys.path.insert(0, os.getcwd())

## Interactive: net house PnL vs model, by demand elasticity

The net house PnL was pre-computed for every elasticity (the simulation is heavy); the slider just
redraws from `assets/10_elasticity_sweep.csv`, so it is instant. The best model at the chosen
elasticity is highlighted in red. Model order and axis are fixed so the bars only change length.
(`pip install ipywidgets` if the slider does not appear.)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

sweep = pd.read_csv("notebooks/assets/10_elasticity_sweep.csv", index_col="elasticity")
model_order = list(sweep.columns)
x_low, x_high = sweep.values.min() * 0.97, sweep.values.max() * 1.02

def plot_for_elasticity(elasticity=8):
    values = sweep.loc[elasticity, model_order]
    best = values.idxmax()
    colors = ["#dc2626" if model == best else "#2563eb" for model in model_order]
    plt.figure(figsize=(9, 4.5))
    plt.barh(model_order, values.values, color=colors)
    plt.gca().invert_yaxis()
    plt.title(f"Net house PnL by model — demand elasticity = {elasticity} (best in red)")
    plt.xlabel("net house PnL")
    plt.xlim(x_low, x_high)
    plt.tight_layout()
    plt.show()

try:
    import ipywidgets as widgets
    widgets.interact(plot_for_elasticity,
                     elasticity=widgets.IntSlider(min=2, max=32, step=1, value=8,
                                                  description="elasticity"))
except ImportError:
    print("ipywidgets not installed; showing elasticity = 2 and 32 instead.")
    plot_for_elasticity(2)
    plot_for_elasticity(32)

## What the slider shows

- The four **vig-only** models are flat in elasticity and clustered around 25.6k-26.3k net PnL
  (they price the direction with the same margin, so demand reacts to them identically).
- **Model 3 swings the most.** At **low elasticity (2)** it is the **best** (~28.0k): demand barely
  reacts to its wider margin, so it keeps volume *and* collects more vig. As elasticity rises its
  volume collapses and it becomes the **worst** (~23.0k at 32). The crossover is around elasticity 7-9.
- The **guarded** model declines mildly with elasticity, staying close to the vig-only cluster.

**Takeaway.** Which model maximises profit depends on how price-sensitive the clients are:
- **Inelastic demand** (clients do not shop on odds) -> the defensive high-margin model 3 wins.
- **Elastic demand** (clients chase the best odds) -> the competitive direction-pricing models win,
  and over-charging (model 3) is the worst choice.

This is exactly why the elasticity is the number to **calibrate from real data**: it decides the
whole pricing strategy. Static preview (sorted within each panel):

![elasticity preview](assets/10_elasticity_sweep.png)

## How the sweep was computed (reproduce)

For each elasticity, each model is run through `simulate` with an elastic `demand_responsive_pool`
plus the predictive and regime-aware arbitrageurs; the base stake is calibrated once so a vig-only
model trades ~200,000 over the window. The cell below regenerates the CSV (a few minutes).

In [ ]:
# Regenerate the sweep (slow). Skip if you only want the slider above.
REGENERATE = False

if REGENERATE:
    from snapmarket.data import load_oracle_prices
    from snapmarket.features import build_features
    from snapmarket.parameters import SharedParameters
    from snapmarket.models import build_model
    from snapmarket.experiments import common_evaluation_start
    from snapmarket.signals import walk_forward_logistic_probability, regime_conditional_probability
    from snapmarket.strategies import demand_responsive_pool, predictive_bettor, regime_aware_bettor
    from snapmarket.engine import simulate

    shared = SharedParameters()
    features = build_features(load_oracle_prices(), shared)
    names = ["momentum_lookup", "momentum_lookup_rolling", "volatility_regime_momentum",
             "momentum_logistic_rolling", "hidden_symmetric_margin",
             "guarded_volatility_regime_momentum"]
    models = {name: build_model(name, features, shared) for name in names}
    start = common_evaluation_start(models.values())
    logistic = walk_forward_logistic_probability(features, shared)
    regime = regime_conditional_probability(features, shared)
    reference_margin, target_volume, window = 0.125, 200_000, 120_000
    reference = simulate(models["volatility_regime_momentum"], features,
                         {"pool": demand_responsive_pool(1.0, 8.0, reference_margin)}, start, window, seed=7)
    base = target_volume / reference.per_bettor["pool"].stake

    def net_pnl(elasticity):
        result = {}
        for name, model in models.items():
            flow = {"pool": demand_responsive_pool(base, elasticity, reference_margin),
                    "predictive": predictive_bettor(logistic, base_stake=base),
                    "regime_aware": regime_aware_bettor(regime, base_stake=base)}
            result[name] = simulate(model, features, flow, start, window, seed=7).house_pnl
        return result

    sweep = pd.DataFrame({e: net_pnl(e) for e in range(2, 33)}).T
    sweep.index.name = "elasticity"
    sweep.to_csv("notebooks/assets/10_elasticity_sweep.csv")